In [8]:
import pandas as pd
import requests
import rasterio
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
import glob
import os
from rasterio.features import rasterize
import numpy as np

raster_folder = "./data"

In [8]:
import re

def get_key_from_environment(file_path: str, key: str) -> str | None:
    """
    Extracts a key from Angular's environment.ts file.

    Args:
        file_path: Path to environment.ts
        key: The key name (e.g., "apiKey")

    Returns:
        The value as a string, or None if not found.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")



In [9]:
header = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

from datetime import datetime
from dateutil.relativedelta import relativedelta

start_date = datetime(2020, 9, 1)   # Sept 2024
end_date = datetime(2025, 8, 1)     # Aug 2025

# Loop over months
date = start_date
while date <= end_date:
    date_str = date.strftime("%Y-%m")
    url = f"https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date={date_str}"
    
    print(f"Fetching {url} ...")
    res = requests.get(url, headers=header)
    
    if res.status_code == 200:
        file = f"./data/temp_{date_str}.tif"
        with open(file, "wb") as f:
            f.write(res.content)
    
    # Move to next month
    date += relativedelta(months=1)

Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2020-09 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2020-10 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2020-11 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2020-12 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2021-01 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2021-02 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2021-03 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2021-04 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=temperature&aggregation=mean&period=month&date=2021

In [10]:
#statewide
shapefile = "../public/shapefiles/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"temp_*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    with rasterio.open(tif) as src:
        nodata = src.nodata
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=nodata
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]*9/5 + 32 
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/temperature/statewide_temperature.csv", index=False)

print(df)

date       state   2020-09    2020-10    2020-11    2020-12    2021-01  \
value  statewide  69.50724  69.106271  67.811977  66.077119  64.395517   

date     2021-02    2021-03    2021-04    2021-05  ...    2024-11    2024-12  \
value  62.873158  63.090849  63.467908  66.361938  ...  66.521916  66.087653   

date     2025-01    2025-02    2025-03    2025-04    2025-05    2025-06  \
value  64.426832  64.294793  64.742593  65.703339  66.312101  68.093021   

date     2025-07    2025-08  
value  69.144584  69.788134  

[1 rows x 61 columns]


In [4]:
folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)

In [10]:
def get_averages_temperature(division, id_col):
  """
  Compute mean temperature per polygon and export to CSV with unique keys like:
  'Hawaiʻi::Kona-1' or 'Maui::Honuaʻula'
  """

  shp_path = f"../public/shapefiles/{division}.shp"
  gdf = gpd.read_file(shp_path).reset_index(drop=True)

  possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni", "isle"]
  island_col = next((c for c in possible_island_cols if c in gdf.columns), None)

  # Combine same names eg Hilo climate division
  # if division in ["county", "climate"]:
  #   if island_col:
  #       gdf = gdf.dissolve(by=[island_col, id_col], as_index=False)
  #   else:
  #       gdf = gdf.dissolve(by=id_col, as_index=False)

  if island_col:
      gdf["division_full"] = gdf.apply(
          lambda r: f"{r[island_col]}::{r[id_col]}" if pd.notna(r[island_col]) else r[id_col],
          axis=1
      )
  else:
      gdf["division_full"] = gdf[id_col]

  # Collect rainfall rasters
  tifs = sorted(glob.glob(os.path.join(raster_folder, "rainfall_*.tif")))
  if not tifs:
      raise FileNotFoundError("No rainfall rasters found")

  # Base raster metadata
  with rasterio.open(tifs[0]) as src:
      raster_crs = src.crs
      transform = src.transform
      shape = (src.height, src.width)

  if gdf.crs != raster_crs:
      gdf = gdf.to_crs(raster_crs)

  # Rasterize polygons
  shapes = [(geom, idx) for idx, geom in enumerate(gdf.geometry)]
  mask = rasterize(shapes, out_shape=shape, transform=transform, fill=-1, dtype="int32")

  # Extract means per time step
  records = []
  for tif in tifs:
      # rainfall_YYYY-MM.tif
      date = os.path.basename(tif).split("_")[1].replace(".tif", "")
      with rasterio.open(tif) as src:
          arr = src.read(1, masked=True)

      for idx, div in enumerate(gdf["division_full"]):
          poly_mask = mask == idx
          vals = arr[poly_mask]
          if np.ma.is_masked(vals):
              vals = vals.compressed()
          mean_val = np.nan if vals.size == 0 else np.nanmean(vals)
          records.append({"division": div, "date": date, "mean_rain": mean_val/25.4})

  # Build tidy table
  df = (
      pd.DataFrame(records)
      .groupby(["division", "date"])["mean_rain"]
      .mean()
      .reset_index()
      .pivot(index="division", columns="date", values="mean_rain")
  )

  df = df.reindex(sorted(df.columns), axis=1)

  out_csv = f"../public/temperature/{division}_temperature.csv"
  df.to_csv(out_csv)
  print(f"Saved {out_csv} ({len(df)} rows)")

In [11]:
get_averages_temperature("island", "name")

Saved ../public/temperature/island_temperature.csv (7 rows)
